# Senior Project Regression Report Demo

## Setup Data

In [1]:
from pathlib import Path

from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mlreport import ComparisonReport, Report

Path("reports").mkdir(exist_ok=True)

X, y = load_diabetes(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"

## Model 1: Train/Test Split Report

In [3]:
split_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Diabetes Regression Report - Train/Test Split",
    author=author,
    description="Linear Regression evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/diabetes_split_report.pdf").to_txt()

Diabetes Regression Report - Train/Test Split
Lucas Summers
Linear Regression evaluated on a standard train/test split.

Model Overview
+-----------------+------------------+
| Property        | Value            |
+=================+==================+
| Name            | LinearRegression |
+-----------------+------------------+
| Type            | Regression       |
+-----------------+------------------+
| Sklearn         | 1.8.0            |
+-----------------+------------------+
| Parameter Count | 13               |
+-----------------+------------------+

Dataset Overview
+--------------------+-------------+
| Property           | Value       |
+====================+=============+
| Features           | 10          |
+--------------------+-------------+
| Total Observations | 442         |
+--------------------+-------------+
| CV Folds           | None        |
+--------------------+-------------+
| Train              | 331 (74.9%) |
+--------------------+-------------+
| Test    

## Model 2: CV Report

In [4]:
cv_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    random_state=42,
)
cv_model.fit(X_train, y_train)

cv_report = Report(
    cv_model,
    title="Diabetes Regression Report - Cross-Validation",
    author=author,
    description="Random Forest evaluated with 5-fold cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/diabetes_cv_report.pdf").to_txt()

Diabetes Regression Report - Cross-Validation
Lucas Summers
Random Forest evaluated with 5-fold cross-validation.

Model Overview
+-----------------+-----------------------+
| Property        | Value                 |
+=================+=======================+
| Name            | RandomForestRegressor |
+-----------------+-----------------------+
| Type            | Regression            |
+-----------------+-----------------------+
| Sklearn         | 1.8.0                 |
+-----------------+-----------------------+
| Parameter Count | 18                    |
+-----------------+-----------------------+

Dataset Overview
+--------------------+---------+
| Property           |   Value |
+====================+=========+
| Features           |      10 |
+--------------------+---------+
| Total Observations |     331 |
+--------------------+---------+
| CV Folds           |       5 |
+--------------------+---------+

Hyperparameters
+--------------------------+---------------+
| Paramet

## Model 3: Tuning + CV Report

In [5]:
search = GridSearchCV(
    estimator=Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor()),
        ]
    ),
    param_grid={
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
    },
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Diabetes Regression Report - Tuning + Cross-Validation",
    author=author,
    description="KNN regressor tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/diabetes_tuned_cv_report.pdf").to_txt()

Diabetes Regression Report - Tuning + Cross-Validation
Lucas Summers
KNN regressor tuned with GridSearchCV and evaluated with 5-fold CV.

Model Overview
+-----------------+---------------------+
| Property        | Value               |
+=================+=====================+
| Name            | KNeighborsRegressor |
+-----------------+---------------------+
| Type            | Regression          |
+-----------------+---------------------+
| Sklearn         | 1.8.0               |
+-----------------+---------------------+
| Parameter Count | 16                  |
+-----------------+---------------------+

Dataset Overview
+--------------------+---------+
| Property           |   Value |
+====================+=========+
| Features           |      10 |
+--------------------+---------+
| Total Observations |     331 |
+--------------------+---------+
| CV Folds           |       5 |
+--------------------+---------+

Hyperparameters
+----------------------+---------------------+
| Para

## Comparison Report of 3 Models

In [6]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Diabetes Workflow Comparison",
    author=author,
    description="Comparison of three regression workflows on the Diabetes dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/diabetes_comparison_report.pdf").to_txt()

Diabetes Workflow Comparison
Lucas Summers
Comparison of three regression workflows on the Diabetes dataset.

Models

Model 1
+-------------+-------------------------------------------------------------+
| Property    | Value                                                       |
+=============+=============================================================+
| Name        | LinearRegression                                            |
+-------------+-------------------------------------------------------------+
| Description | Linear Regression evaluated on a standard train/test split. |
+-------------+-------------------------------------------------------------+
| Type        | Regression                                                  |
+-------------+-------------------------------------------------------------+
| Data        | Train/Test Split                                            |
+-------------+-------------------------------------------------------------+
| Params      | 